# 00 · Engenharia de Dados - MVP - Qualidade das distribuidoras de energia elétrica de grande porte do Brasil

**Avaliação da evolução da qualidade das distribuidoras de energia elétrica de grande porte do Brasil**

Pós-graduação em Ciência de Dados e Analytics (PUC-Rio) · Sprint 1 — Engenharia de Dados

Repositório: https://github.com/gaca-ai/puc-mvp-01-performance-dx

---

Este notebook não executa código. Ele define o problema, as perguntas que o pipeline deve responder, o pipeline adotado e o caminho que os dados percorrem até virarem resposta.

A ordem não é acidental. Um pipeline construído sem clareza do destino é um pipeline que precisa ser refeito: sem saber o que se quer responder, não há como decidir quais fontes buscar, qual granularidade preservar nem como modelar as tabelas.

## 1. Contexto

A distribuição de energia elétrica no Brasil é um monopólio natural. Em cada área de concessão existe uma única distribuidora, e o consumidor não pode trocar de fornecedor se o serviço for ruim. Não havendo concorrência para disciplinar a qualidade, esse papel cabe à regulação.

A ANEEL (Agência Nacional de Energia Elétrica) exerce esse papel exigindo que cada distribuidora apure e informe indicadores padronizados de continuidade, tensão, atendimento comercial e investimento. Esses dados são publicados no portal de dados abertos da Agência.

Existe, portanto, uma base pública que permite comparar o desempenho das distribuidoras sem depender de informação interna das empresas. É essa base que o MVP explora.

**A escolha analítica central deste trabalho é medir evolução, não nível.**

Comparar o nível absoluto de qualidade entre distribuidoras é enganoso: uma empresa que atende uma região metropolitana densa, com rede subterrânea, parte de uma condição estruturalmente diferente de outra que atende área rural extensa, com rede aérea exposta a vegetação e descargas atmosféricas. Comparar o nível é, em boa medida, comparar geografia.

A evolução é mais justa: cada distribuidora é comparada consigo mesma no tempo, e o ranking mede quem mais melhorou a partir da própria condição inicial.

### Por que agora

O tema ganhou peso institucional recente. Dezenove concessões de distribuição têm contratos vencendo entre 2025 e 2031, e o Decreto nº 12.068/2024 definiu as regras para prorrogá-las por mais 30 anos, condicionando a renovação à aceitação de novas obrigações de qualidade, eficiência, gestão econômico-financeira e modernização das redes. A ANEEL aprovou o termo aditivo correspondente em fevereiro de 2025 e, ao longo daquele ano, emitiu pareceres favoráveis à renovação de onze concessões.

Em abril de 2026, o Ministério de Minas e Energia prorrogou por 30 anos os contratos de catorze distribuidoras. As três concessões da Enel — Ceará, Rio de Janeiro e São Paulo — ficaram fora da lista, mesmo com a ANEEL tendo recomendado a renovação das operações cearense e fluminense, e a concessão paulista passou a enfrentar processo de caducidade.

Isso torna a pergunta deste MVP concreta em vez de acadêmica. O histórico recente de qualidade deixou de ser apenas um indicador de acompanhamento e passou a ter consequência contratual direta. Analisar quem melhorou nos últimos três anos é, em boa medida, analisar o período que antecedeu e informou essas decisões.

Há um efeito colateral analítico que precisa ficar registrado. Um ciclo de renovação cria incentivo para que as empresas melhorem indicadores e antecipem investimentos justamente na janela em que serão avaliadas. Parte da evolução observada pode refletir esse esforço concentrado, e não uma melhora estrutural sustentável. O pipeline não consegue separar as duas coisas, mas a interpretação dos resultados deve levar essa possibilidade em conta.

## 2. Problema

> Entre as concessionárias de grande porte do Brasil, quais apresentaram a melhor evolução na qualidade percebida pelo consumidor desde 2022?

O termo **qualidade percebida pelo consumidor** delimita o escopo. O recorte é o que o cliente sente na ponta: quanto tempo ficou sem energia, com que frequência, se a tensão entregue está adequada, se o serviço solicitado foi cumprido no prazo e se ele precisou reclamar.

Fica de fora o desempenho operacional interno, como produtividade de equipes ou eficiência de custos, que não é observável pelo consumidor.

### O que este trabalho acrescenta

A ANEEL já apura e publica os indicadores de qualidade, e já os disponibiliza calculados, inclusive em painéis de BI. Reproduzir esses números não seria contribuição alguma.

O que este trabalho propõe é outra coisa: métricas redefinidas para avaliar a distribuidora sob a ótica do consumidor, e não sob a ótica da penalidade regulatória. São coisas diferentes, porque o indicador regulatório é construído para ser justo com a empresa na hora de punir, e isso o obriga a descontar eventos excepcionais e a incluir atendimentos que o cliente nunca sentiu.

Esse princípio é o que explica as duas escolhas metodológicas centrais, registradas na seção 5, e é o critério para resolver qualquer dúvida futura de escopo: mede-se o que o consumidor sente.

## 3. Perguntas de negócio

**Evolução por dimensão**

1. Quais distribuidoras mais reduziram as reclamações procedentes por mil unidades consumidoras?
2. Quais mais aumentaram o percentual de serviços comerciais realizados dentro do prazo regulatório?
3. Quais mais reduziram o DEC e o FEC por falha interna?
4. Quais mais reduziram a transgressão dos limites de nível de tensão (DRP e DRC)?
5. Quais mais investiram em melhoria e renovação da rede por unidade consumidora?
6. Quais mais reduziram o tempo médio de atendimento a ocorrências com interrupção (TMA-CI)?

**Relações entre dimensões**

7. Maior investimento por unidade consumidora está associado a maior melhora de continuidade?
8. A redução de DEC e FEC se reflete em menos reclamações por falta de energia?
9. Qual o peso das interrupções que o indicador deixa de fora — programadas e de origem externa — na indisponibilidade total sentida pelo consumidor?
10. Há distribuidoras com DEC e FEC por falha interna em queda e TMA-CI em alta ao mesmo tempo? Esse par de sinais é compatível com automação de redes, e precisa ser lido em conjunto.

As perguntas 1 a 6 produzem os seis rankings. As perguntas 7 a 10 são as que transformam rankings em interpretação: elas testam se investir melhora o serviço, se o serviço melhor reduz o atrito com o cliente, quanto da indisponibilidade fica fora do indicador escolhido, e se a combinação dos dois sinais técnicos revela mudança na forma de operar a rede.

> Conforme a especificação do trabalho, estas perguntas permanecem registradas mesmo que alguma não seja respondida. A autoavaliação discutirá o que foi e o que não foi atingido.

## 4. Glossário

| Sigla | Significado | O que mede |
|---|---|---|
| UC | Unidade Consumidora | O ponto de entrega de energia, na prática o cliente |
| DEC | Duração Equivalente de Interrupção por Unidade Consumidora | Quantas horas, em média, cada cliente ficou sem energia no período |
| FEC | Frequência Equivalente de Interrupção por Unidade Consumidora | Quantas vezes, em média, cada cliente ficou sem energia no período |
| DRP | Duração Relativa da Transgressão de Tensão Precária | Percentual do tempo com tensão fora da faixa adequada |
| DRC | Duração Relativa da Transgressão de Tensão Crítica | Percentual do tempo com tensão em faixa crítica |
| — | Ocorrência emergencial | Atendimento de emergência provocado por um único evento que gere deslocamento de equipes, inclusive quando a causa for considerada improcedente (PRODIST, Módulo 1, item 249). O critério é o deslocamento da equipe, não a existência de interrupção |
| TMAE | Tempo Médio de Atendimento a Emergências | Indicador regulatório. Quantos minutos, em média, a distribuidora leva para atender uma ocorrência emergencial, somando preparação da equipe (TMP), deslocamento até o local (TMD) e execução até o restabelecimento (TME), sobre todas as ocorrências do período |
| TMA-CI | Tempo Médio de Atendimento a Ocorrências com Interrupção | Indicador próprio deste trabalho. Mesma soma de parcelas do TMAE, mas apurada somente sobre as ocorrências que causaram interrupção de fornecimento, excluídas as programadas e as de origem externa ao sistema de distribuição. Não é o TMAE e não deve ser comparado a valores publicados do TMAE |
| DEC-FI e FEC-FI | DEC e FEC por Falha Interna | Indicadores próprios deste trabalho. Somam as parcelas de origem interna e não programada do DEC e do FEC, incluindo ISE e Dia Crítico: `DECind + DECine + DECinc`. Não são o DEC e o FEC comparados com os limites regulatórios |
| DECip, DECind | Parcelas do DEC apuradas contra o limite | `DECip` é a parcela de origem interna e programada fora de Dia Crítico; `DECind`, a de origem interna, não programada e não expurgável. O indicador regulatório é a soma das duas — ou seja, já inclui manutenção programada |
| DECine, DECinc, DECino | Parcelas expurgáveis de origem interna | `DECine` é a parcela de ISE; `DECinc`, a de Dia Crítico; `DECino`, a de racionamento instituído pela União e de atuação do ERAC. As duas primeiras entram no DEC-FI; a terceira não |
| DECxp, DECxn | Parcelas de origem externa | Interrupções originadas fora do sistema de distribuição, programadas e não programadas. Ficam fora do DEC-FI |
| ISE | Interrupção em Situação de Emergência | Interrupção ocorrida em situação de calamidade reconhecida por decreto. A regulação a expurga do DEC; este trabalho a mantém |
| PNIE | Percentual do Número de Ocorrências Emergenciais com Interrupção de Energia | Que fração das ocorrências atendidas no período envolveu interrupção. Aqui tem um segundo papel: mede a cobertura do recorte do TMA-CI, isto é, quanto do universo de atendimentos permanece no indicador |
| DGC | Desempenho Global de Continuidade | Critério do ranking oficial da ANEEL; média das razões entre valor apurado e limite de DEC e de FEC |
| SAC | Serviço de Atendimento ao Cliente | Primeiro nível de atendimento da distribuidora |
| INDGER | Sistema de Indicadores Gerenciais | Base em que as distribuidoras informam dados técnicos e comerciais à ANEEL |
| PDD | Plano de Desenvolvimento da Distribuição | Plano anual de investimentos em expansão, melhoria e renovação da rede |
| PRODIST | Procedimentos de Distribuição | Conjunto de normas técnicas da ANEEL; o Módulo 8 trata de qualidade |
| Conjunto | Conjunto de Unidades Consumidoras | Agrupamento geográfico usado pela ANEEL para apurar continuidade |

## 5. Recorte

| Dimensão | Decisão | Justificativa |
|---|---|---|
| Universo | Concessionárias com mais de 400 mil UCs | Mesmo corte de porte que a ANEEL usa no ranking oficial de continuidade; distribuidoras pequenas têm realidade operacional distinta |
| Identificação | CNPJ, e não sigla | Siglas mudam com aquisições e renomeações, o que quebraria a série histórica |
| Janela | Série a partir de 2022 | 2020 e 2021 ficam fora por serem atípicos em razão da pandemia, e 2022 é o primeiro ano do regime atual de composição do DEC/FEC. A forma de agregação — ano civil ou janela móvel de doze meses — é definida por métrica, conforme o comportamento do dado |
| Comparação | Bloco final (últimos 12 meses) contra bloco inicial (12 meses iniciais) | Comparar períodos de 12 meses elimina a sazonalidade, como o efeito do período chuvoso sobre as interrupções |
| Resultado | Seis rankings independentes | Um índice composto único exigiria arbitrar pesos entre dimensões não comparáveis, e esconderia que uma empresa pode melhorar muito em uma frente e piorar em outra |
| Sinal | Evolução positiva sempre significa melhora | Algumas métricas melhoram caindo, como o DEC, e outras subindo, como o percentual no prazo |

**Uma nota sobre o DEC e o FEC.** O ranking não usa o indicador regulatório nem a soma de todas as parcelas. Usa um indicador próprio, o **DEC-FI**, composto pelas parcelas de origem interna e não programada, incluindo as que a regulação expurga:

$$DEC\text{-}FI = DEC_{ind} + DEC_{ine} + DEC_{inc}$$

Ficam de fora as parcelas programadas (`DECip` e `DECipc`), as de origem externa ao sistema de distribuição (`DECxp` e `DECxn`) e a parcela decorrente de racionamento e de atuação do ERAC (`DECino`). O FEC-FI segue a mesma composição.

Duas decisões separam esse indicador do regulatório, e elas vão em sentidos opostos.

A primeira é **incluir o que a regulação expurga**: ISE e Dia Crítico. O expurgo é correto para fins de penalidade, porque não faz sentido multar a empresa por um temporal excepcional. Mas o consumidor que ficou sem energia num temporal ficou sem energia, e a pergunta deste trabalho é sobre o que ele sentiu.

A segunda é **excluir o que a regulação inclui**: a manutenção programada, que compõe o DEC oficial pela parcela `DECip`. O argumento não é que o consumidor não sentiu — ele sentiu. O argumento é que essa interrupção é de natureza diferente, por três razões.

Ela é previsível e comunicada. O Módulo 8 exige aviso prévio, e o consumidor pode se organizar. Uma indisponibilidade anunciada e uma falha inesperada produzem prejuízos distintos, e somá-las trata como equivalente o que não é.

Ela não é falha do serviço, é condição de mantê-lo. Substituir um poste deteriorado ou reconfigurar um alimentador exige desligar o trecho. A interrupção não sinaliza degradação da rede; sinaliza que a rede está sendo conservada.

E, sobretudo, incluí-la produziria incentivo contrário ao que este trabalho quer medir. Uma distribuidora que adia manutenção reduz `DECip` no curto prazo e sobe no ranking, enquanto a que investe em conservar a rede é penalizada — exatamente a empresa que a métrica 5 premia por investir em melhoria e renovação. Um indicador que recompensa o adiamento da manutenção mede o inverso do que pretende medir.

Há uma vulnerabilidade nessa escolha, e ela fica registrada. Como a parcela programada sai do indicador, uma distribuidora poderia classificar como programada uma indisponibilidade que não é, deslocando duração para fora da medição. Por isso a parcela `DECip` é acompanhada como métrica descritiva, fora do ranking, no mesmo papel que o PNIE cumpre para o TMA-CI: crescimento anômalo da fatia programada é sinal a investigar, não resultado a celebrar.

O critério de ranking segue o espírito do DGC ao dar peso igual a DEC e FEC, mas mede a variação de cada um em vez da razão contra o limite regulatório. O motivo é que os limites são redefinidos nas revisões tarifárias: uma empresa poderia cair no ranking sem ter piorado, apenas porque o limite ficou mais rígido. A reconciliação com o DGC oficial é feita à parte, como teste de qualidade do pipeline.

**Uma nota sobre o tempo de atendimento.** O indicador entra como ranking próprio, e não somado ao DEC e ao FEC, porque mede outra coisa: continuidade é a qualidade do produto entregue, tempo de atendimento é a qualidade da resposta quando o produto falha. Juntá-los num indicador único exigiria arbitrar um peso entre grandezas sem denominador comum.

O indicador regulatório é o TMAE, e ele não serve ao recorte deste trabalho. A razão está na definição de ocorrência emergencial, no Módulo 1 do PRODIST, item 249: "atendimento de emergência provocado por um único evento que gere deslocamento de equipes, inclusive quando a causa for considerada improcedente". O critério é o deslocamento da equipe. Um chamado improcedente, em que a equipe foi e não encontrou problema na rede, conta no TMAE com o mesmo peso de uma interrupção real — e o consumidor não ficou sem energia em momento algum.

Por isso este trabalho usa o TMA-CI, tempo médio de atendimento a ocorrências com interrupção: mesma estrutura de parcelas do TMAE, mas apurado somente sobre as ocorrências que efetivamente interromperam o fornecimento. O princípio é o mesmo que define o DEC-FI — mede-se a falha não planejada do fornecimento, e o critério de exclusão é idêntico nos dois indicadores.

$$TMA\text{-}CI = \frac{\sum (TP + TD + TE)}{\text{número de ocorrências com interrupção}}$$

Como o denominador do indicador é a contagem de ocorrências, e não a quantidade de clientes, a soma pode ser feita direto no nível da distribuidora. Não há ponderação por unidade consumidora: ponderar por UC aqui, por analogia com o DEC e o FEC, produziria um número sem significado.

O critério de exclusão é o mesmo do DEC-FI, aplicado ao universo de ocorrências. Ficam fora as de fato gerador programado e as de origem externa ao sistema de distribuição, pelas razões já expostas. A ISE permanece, também pela mesma razão: o consumidor ficou sem energia.

Ficam fora, ainda, as ocorrências de defeito interno da própria unidade consumidora. A distribuidora não tem como resolvê-las — a causa é falha de manutenção da instalação do cliente — e o alcance é de um cliente, não do conjunto. E ficam fora os atendimentos que o próprio Módulo 8 exclui da apuração por não serem emergência de fornecimento: iluminação pública, serviços de caráter comercial e reclamações de nível de tensão.

O PNIE acompanha o ranking como métrica descritiva, com dois papéis: mostrar a composição dos atendimentos de cada empresa e medir que fração do universo o recorte do TMA-CI preserva.

**Uma nota sobre automação de redes.** Como a ocorrência emergencial só existe quando há deslocamento de equipe, a interrupção restabelecida remotamente, por telecomando ou religador automático, não entra no indicador. Uma distribuidora que automatiza a rede retira do denominador justamente os casos mais simples e rápidos, e o TMA-CI pode subir enquanto a qualidade entregue melhora. Isso não é defeito da métrica, é informação: DEC-FI e FEC-FI em queda acompanhados de TMA-CI em alta são um par de sinais compatível com automação, e a análise de continuidade trata essa leitura conjunta.

**Uma nota sobre validação.** Todo indicador calculado aqui é reconciliado contra o valor equivalente publicado pela ANEEL, quando existe publicação. No caso do tempo de atendimento, isso significa calcular também o TMAE no universo completo, para comparar com o indicador oficial. A reconciliação tem duas funções: provar que o pipeline reproduz o número oficial quando aplica a regra oficial, e medir quanto cada escolha metodológica própria desloca o resultado. Se o ranking muda pouco, a escolha é robusta; se muda muito, a diferença é, por si, um achado.


## 6. Fontes de dados

Todas as bases vêm do portal de dados abertos da ANEEL, em https://dadosabertos.aneel.gov.br, coletadas pela API do CKAN. A licença de cada conjunto é registrada na etapa de coleta.

| Fonte | Granularidade | Usada para |
|---|---|---|
| Indicadores Coletivos de Continuidade | conjunto × período × indicador | Parcelas do DEC e do FEC, das quais se compõem o DEC-FI e o FEC-FI |
| Conformidade do Nível de Tensão | UC sorteada × ano × indicador | Transgressões de DRP e DRC |
| INDGER — Dados Comerciais | município × mês | Quantidade de UCs ativas, usada como denominador e como filtro de porte |
| INDGER — Dados de Serviços Comerciais | município × tipo de serviço × mês | Serviços realizados dentro e fora do prazo |
| Manifestações no 1º e 2º níveis | município × canal × tipologia × mês | Reclamações procedentes |
| Plano de Desenvolvimento da Distribuição | CNPJ × UF × ano × tipo de obra | Investimento realizado |
| Ocorrências Emergenciais nas Redes de Distribuição | ocorrência | Parcelas de tempo, fato gerador e interrupção associada, base do TMA-CI e do PNIE |
| Atendimento às Ocorrências Emergenciais | conjunto × mês × indicador | Apenas reconciliação: TMAE, TMP, TMD, TME e PNIE já apurados pela ANEEL |

Complementam essas bases algumas tabelas de referência de autoria própria, derivadas de dados públicos, que resolvem correspondências entre códigos: histórico de CNPJ e sigla, tipologias de reclamação e códigos de serviço.

**Uma observação sobre granularidade.** Nenhuma fonte está no nível em que a pergunta é feita. A continuidade vem por conjunto, o comercial por município, a tensão por unidade consumidora sorteada, o investimento por obra e as ocorrências emergenciais evento por evento. Todas precisam ser elevadas ao nível distribuidora-período, e essa consolidação tem regra própria em cada caso. O DEC e o FEC, por exemplo, são médias ponderadas pela quantidade de clientes de cada conjunto, e não médias simples. Boa parte do trabalho de engenharia deste MVP está aí.

O atendimento emergencial é o caso mais sutil dessa consolidação, e por isso está detalhado na seção anterior: o denominador é a contagem de ocorrências, não a quantidade de clientes, e por isso ele não sobe por média ponderada como o DEC e o FEC.

**Uma observação sobre volume.** A base de ocorrências emergenciais é a maior do conjunto, com uma linha por evento para todo o país, e o projeto roda na Free Edition do Databricks, que tem cota diária de processamento. Quando o download direto pela API não couber na cota, o arquivo é baixado e filtrado localmente, restringindo aos CNPJs do universo e às colunas usadas, e o recorte é carregado no volume de landing. A etapa manual fica registrada na documentação do pipeline, com o script de filtragem versionado no repositório, de modo que o caminho continue reproduzível.

## 7. Arquitetura do pipeline

O pipeline segue a arquitetura medalhão, com três camadas de refinamento progressivo, implementadas como schemas do catálogo `mvp_aneel` no Unity Catalog.

```
Portal ANEEL (API CKAN)
        |
        v
  Volume de landing          arquivos originais preservados
        |
        v
      BRONZE                 dados como vieram, mais metadados de ingestao
        |
        v
      SILVER                 tipados, deduplicados, padronizados, no nivel distribuidora-periodo
        |
        v
       GOLD                  dimensoes, fatos, metricas de evolucao e rankings
        |
        v
     Analise                 respostas as perguntas da secao 3
```

| Camada | Responsabilidade | O que não acontece aqui |
|---|---|---|
| Bronze | Preservar a evidência original e registrar quando e de onde cada dado veio | Nenhuma correção, nem de tipo, nem de duplicata |
| Silver | Limpar, tipar, deduplicar, padronizar chaves e aplicar o recorte de escopo | Nenhum cálculo de métrica de negócio |
| Gold | Modelar e calcular métricas e rankings | Nenhuma limpeza; se sujeira chegou até aqui, o erro está na Silver |

A separação existe para tornar o erro rastreável. Quando um número final parece estranho, é possível descer camada por camada até encontrar onde ele se deformou, e a Bronze garante que o dado original continua disponível para conferência.

## 8. Roteiro dos notebooks

| Notebook | O que faz |
|---|---|
| `00_objetivo` | Este documento: problema, perguntas, recorte e arquitetura |
| `01_setup` | Cria catálogo, schemas e volume; testa o acesso ao portal da ANEEL |
| `02_bronze_ingestion` | Baixa os arquivos e carrega as tabelas Bronze com metadados de ingestão |
| `03_bronze_data_quality` | Perfila cada atributo: completude, consistência, unicidade, acurácia e outliers |
| `04_silver_reference` | Trata as tabelas de referência e os de-para de códigos |
| `05_silver_technical` | Consolida continuidade, nível de tensão e ocorrências emergenciais no nível distribuidora-período |
| `06_silver_commercial` | Consolida UCs ativas, serviços comerciais e reclamações |
| `07_silver_investment` | Trata o PDD e deflaciona os valores |
| `08_gold_model` | Constrói as dimensões e as tabelas fato |
| `09_gold_rankings` | Calcula as métricas de evolução e os seis rankings; é onde cada indicador próprio é implementado e diferenciado do indicador regulatório correspondente |
| `10_analysis` | Responde às perguntas e reconcilia os indicadores contra os valores publicados pela ANEEL |
| `11_data_catalog` | Documenta tabelas e campos no Unity Catalog |

**Uma nota sobre a autoavaliação.** A especificação do trabalho pede uma autoavaliação ao final. Aqui ela é distribuída: cada notebook encerra com uma seção própria, avaliando o que aquela etapa entregou, o que não saiu como planejado e o que ficou em aberto. A autoavaliação do documento final consolida essas seções.

A razão é prática. Escrever a avaliação junto da etapa preserva o que se aprendeu enquanto o problema ainda está na cabeça; reconstruí-la no fim, olhando dez notebooks para trás, produz um texto genérico. E a dificuldade que aparece na Silver tem natureza diferente da que aparece na modelagem — separá-las torna a avaliação verificável, porque cada afirmação fica ao lado da evidência que a sustenta.

---

## 9. Autoavaliação desta etapa

**O que esta etapa entregou.** O problema delimitado, as dez perguntas de negócio, o recorte de universo e janela, a definição dos dois indicadores próprios — DEC-FI e TMA-CI — com a justificativa de cada afastamento em relação ao indicador regulatório correspondente, e a arquitetura do pipeline.

**O que mudou no caminho.** A definição de continuidade começou como "DEC total, sem expurgo" e não resistiu à leitura do Módulo 8. Descobriu-se que o DEC oficial já inclui manutenção programada, de modo que a escolha inicial não era "incluir o que a regulação exclui", como o texto afirmava, mas apenas isso mais a manutenção programada por inércia. A revisão produziu um critério explícito, simétrico entre continuidade e atendimento, e obrigou a nomear o indicador — DEC-FI — em vez de chamá-lo de total, que ele não é.

**O que fica em aberto.** Os indicadores estão definidos a partir do dicionário de dados e da norma, não dos dados. A verificação de que cada parcela existe, está preenchida e é consistente ao longo da janela é trabalho da etapa de qualidade, e pode obrigar a rever o que aqui ficou decidido.

**O que eu faria diferente.** Ler o Módulo 8 antes de redigir o recorte, e não depois. A definição de continuidade foi escrita sobre uma suposição razoável e errada, e a correção custou uma revisão que teria sido evitada por uma leitura de quinze minutos na ordem certa.

---

**Próximo passo:** `01_setup`.